# Phase 1 Pipeline: Python Trace Generation & Training

This pipeline notebook orchestrates the entirety of Phase 1 for the World Model: generating deterministic multi-line Python traces with explicit data dependencies, training a quantized LoRA adapter on Qwen 2.5, and performing rigorous state-fidelity evaluation.

In [ ]:
# Clone the GitHub repository and navigate into it so relative paths work perfectly in Colab
!git clone https://github.com/TinevimboMusingadi/lm-world-model.git
%cd lm-world-model

## 1. Setup & Dependencies
Install requirements tailored for Colab free tier GPU limitations.

In [1]:
!pip install -q transformers datasets trl peft bitsandbytes wandb jsonlines pydantic

import torch
import transformers
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00:00:0100:01
PyTorch: 2.9.0+cu126 | CUDA: True


## 2. Generate Trace Dataset
Our data generator is protected against trivial memorization. Variables traverse a wide range of values, and rigorous AST routing enforces data dependency chains.

In [ ]:
!python -m phase1_python.generate_dataset --seed 42 --n 10000 --output data/splits/phase1_raw.jsonl
!python -m data.inspector --file data/splits/phase1_raw.jsonl

### Dataset Visualizations
Let's visualize the trace length distributions and program complexities to understand our data better before training.

In [ ]:
import pandas as pd
import jsonlines
import plotly.express as px

records = []
with jsonlines.open('data/splits/phase1_raw.jsonl') as f:
    for r in f:
        records.append({'complexity': str(r['complexity']), 'trace_len': len(r['execution_trace'].split('\n'))})

df = pd.DataFrame(records)
fig = px.histogram(df, x='trace_len', color='complexity', 
                   title='Trace Length Distribution by Complexity Level', 
                   barmode='overlay', nbins=50)
fig.show()

## 3. PEFT LoRA Training
We fine-tune Qwen2.5-0.5B on the trace data using 4-bit NormalFloat and targeting attention projectors.

In [ ]:
import yaml
from pathlib import Path

config = {
    "run_name": "phase1-qwen0.5b-condB-seed42",
    "phase": "phase1",
    "condition": "B",
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "data_file": "data/splits/phase1_raw.jsonl",
    "output_dir": "checkpoints/phase1_qwen0.5b_condB",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 1,
    "batch_size": 4,
    "grad_accum": 4,
    "lr": 2.0e-4,
    "max_seq_len": 1024
}
Path("training/configs").mkdir(parents=True, exist_ok=True)
with open("training/configs/phase1_qwen_0.5b.yaml", "w") as f:
    yaml.dump(config, f)

!python -m training.train_sft --config training/configs/phase1_qwen_0.5b.yaml

## 4. Robust Evaluation
Inference tests including Levenshtein edit distance and $T=0.5$ sampling checks to discover if testing fragility stems from overfitting or structural flaws.

In [ ]:
!python -m eval.evaluate --checkpoint checkpoints/phase1_qwen0.5b_condB \
                         --data data/splits/phase1_raw.jsonl \
                         --split test_indist \
                         --condition B \
                         --temperature 0.0

!python -m eval.evaluate --checkpoint checkpoints/phase1_qwen0.5b_condB \
                         --data data/splits/phase1_raw.jsonl \
                         --split test_indist \
                         --condition B \
                         --temperature 0.5

### Evaluation Visualizations
Visualizing the robustness of the trained model when sampled at different temperatures (Temperature tests from 'thing_to_out_for.md').

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    df_res = pd.read_csv("results/eval_results.csv")
    df_p1 = df_res[df_res['split'] == 'test_indist'].copy()
    df_p1['temperature_str'] = df_p1['temperature'].astype(str)
    
    fig = px.bar(df_p1, x="temperature_str", y=["output_accuracy", "trace_accuracy"], 
                 barmode="group", 
                 title="Robustness Degradation: Evaluated Accuracy vs Prediction Temperature",
                 labels={"value": "Accuracy (%)", "temperature_str": "Sampling Temperature"})
    fig.show()
except FileNotFoundError:
    print("Run the evaluate cells above first to generate the csv.")